In [1]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig

#model_path = "meta-llama/Llama-3.2-1B"
model_path = "google/gemma-3-1b-pt"
tokenizer = AutoTokenizer.from_pretrained(model_path)
#tokenizer = AutoTokenizer.from_pretrained("gpt2")

/Users/pariidan/.pyenv/versions/finetuning_fin_llms/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sentence1 = "The cat sat on the mat"
sentence2 = "The dog ate my homework"
sentence3 = "My aunt is a teacher"


sentences = [sentence1, sentence2, sentence3]
tokenized_sentences = tokenizer(sentences, return_attention_mask=False, add_special_tokens= False)['input_ids']
tokenized_sentences = [t for s in tokenized_sentences for t in s + [tokenizer.eos_token_id]]
tokenizer.decode(tokenized_sentences)

'The cat sat on the mat<eos>The dog ate my homework<eos>My aunt is a teacher<eos>'

In [ ]:
### issue is that we need the mask to be w.r.t to examples within the tensor, such that whilst we calculate everything in parallel we don't just waste a lot of the sequnces with padded operations

def get_attention_for_packed_sequence(x, token_id, eos: bool = True):
    # store seq length
    T = tokenized_sentences.size(0)

    # get indices of all EOS tokens
    eos_indices = (tokenized_sentences == tokenizer.eos_token_id).nonzero().squeeze()

    # get lengths of all sequences
    reps = torch.cat([eos_indices[[0]] + 1, eos_indices[1:] - eos_indices[:-1]])

    # repeat each eos token n times along dim 1, where n is number of tokens per subsequence
    repeated_idx = torch.repeat_interleave(eos_indices, reps).view(1,-1).expand(T,-1)

    # create square matrix of of size T,T with indices from 0 to T
    mask_indices = torch.arange(T).view(-1, 1).expand(-1, T)

    # create traditioanl attn matrix
    mask = torch.ones(T,T,dtype = torch.int).tril()

    # mask out all tokens frim preceding sequences
    mask.masked_fill_(mask_indices > repeated_idx, False)

    # pos_ids = torch.arange(T) - torch.repeat_interleave(torch.cat([torch.tensor([0]), eos_indices+1], dim=0)[:-1], reps)
    # pos_ids

    return mask

mask_attn = get_attention_for_packed_sequence(tokenized_sentences, tokenizer.eos_token_id)

In [8]:
sentence4 = "Rome wasn't built in a day"
sentence5 = "My hovercraft is full of eels"

sentences = [sentence4, sentence5]
tokenized_sentences2 = tokenizer(sentences, return_attention_mask=False, add_special_tokens=False)["input_ids"]
tokenized_sentences2 = torch.tensor([t for s in tokenized_sentences2 for t in s + [tokenizer.eos_token_id]])

batch = torch.nn.utils.rnn.pad_sequence(
  [tokenized_sentences, tokenized_sentences2],
  batch_first=True, padding_value=tokenizer.eos_token_id
)

In [9]:
B, T = batch.shape

In [10]:
eos_idx = (batch.view(-1) == tokenizer.eos_token_id).nonzero(as_tuple=True)[0] + 1
eos_idx_expanded = torch.cat(
  [eos_idx, torch.arange(0,B*T+1,T)]
).unique().sort()[0]

In [11]:
normalized_idx = eos_idx_expanded - (eos_idx_expanded // T) * T
normalized_idx = torch.where(normalized_idx == 0, T, normalized_idx)

In [12]:
reps = normalized_idx[1:] - normalized_idx[:-1]
reps = torch.where(reps < 1, normalized_idx[1:], reps)

In [ ]:
def get_attention_mask_for_packed_sequence(x, token_id, eos: bool = True):
    B, T = x.shape
    eos_idx = (x.view(-1) == token_id).nonzero(as_tuple=True)[0] + eos
    eos_idx_expanded = torch.cat([eos_idx, torch.arange(0,B*T+1,T)]).unique().sort()[0]
    normalized_idx = eos_idx_expanded - (eos_idx_expanded // T) * T
    normalized_idx = torch.where(normalized_idx == 0, T, normalized_idx)
    reps = normalized_idx[1:] - normalized_idx[:-1]
    reps = torch.where(reps < 1, normalized_idx[1:], reps)
    repeated_idx = torch.repeat_interleave(normalized_idx[1:], reps).view(B,1,T).expand(-1,T,-1)
    mask_indices = torch.arange(T).view(1,-1,1).expand(B, -1, T)
    mask = torch.ones(T, T, dtype=torch.bool).tril().expand(B, -1, -1)
    mask = mask.masked_fill(mask_indices >= repeated_idx, False)
    # get position ids for packed sequence
    pos_ids = (torch.arange(B*T) - torch.repeat_interleave(eos_idx_expanded[:-1], reps)).view(B,T)
    return mask, pos_ids

In [ ]:
mask, pos_ids = get_attention_mask_for_packed_sequence(batch, tokenizer.eos_token_id)

In [16]:
### using it wthin a DataCollator: https://chatgpt.com/c/691b8483-90a0-8321-bc25-3ab31d341cbd

## Test new DataCollator

In [28]:
%pip show bitsandbytes
%pip install -U bitsandbytes
%pip show bitsandbytes
%pip show transformers

Name: bitsandbytes
Version: 0.48.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy, packaging, torch
Required-by: unsloth
Name: bitsandbytes
Version: 0.48.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy, packaging, torch
Required-by: unsloth
Name: transformers
Version: 4.57.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contribut

In [29]:
import os

In [30]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

#model_path = "meta-llama/Llama-3.2-1B"
model_path = "google/gemma-3-1b-pt"

model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

KeyboardInterrupt: 

In [31]:
# unsloth version
from unsloth import FastLanguageModel

max_seq_len = 2048
dtype = None
load_in_4bit = True
unsloth_path = "google/gemma-3-1b-pt"
unsloth_path = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = unsloth_path,
    max_seq_len = max_seq_len,
    dtype = None,
    load_in_4bit = load_in_4bit
)

==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [6]:
# from datasets import Dataset
# ds = Dataset.load_from_disk('../data/formatted_data/norobots').select(range(100))

from datasets import load_dataset
ds = load_dataset('danp27/norobots_sft')['train'].select(range(100))

README.md:   0%|          | 0.00/533 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/21.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9999 [00:00<?, ? examples/s]

In [7]:
reserved = [
    tok.content
    for tok in tokenizer.added_tokens_decoder.values()
    if "reserved_special_token" in tok.content
]

In [8]:
# ---- Special token maps ----

LLAMA_SPECIAL_MAP = {
    "<sistem>": "<|reserved_special_token_0|>",
    "</sistem>": "<|reserved_special_token_1|>",
    "<utilizator>": "<|reserved_special_token_2|>",
    "</utilizator>": "<|reserved_special_token_3|>",
    "<asistent>": "<|reserved_special_token_4|>",
    "</asistent>": "<|reserved_special_token_5|>",
}

GEMMA_SPECIAL_MAP = {
    "<sistem>": "<unused0>",
    "</sistem>": "<unused1>",
    "<utilizator>": "<unused2>",
    "</utilizator>": "<unused3>",
    "<asistent>": "<unused4>",
    "</asistent>": "<unused5>",
}

SPECIAL_MAPS = {
    "llama": LLAMA_SPECIAL_MAP,
    "gemma": GEMMA_SPECIAL_MAP,
}


# ---- Model family detection ----

def detect_model_family(tokenizer):
    vocab = tokenizer.get_vocab()

    # LLaMA-style special reserved tokens
    if "<|reserved_special_token_0|>" in vocab:
        return "llama"

    # Gemma-style unused tokens
    if "<unused0>" in vocab:
        return "gemma"

    return None  # fallback if neither is detected


# ---- Generic tag replacer ----

def replace_tags(text, tag_map):
    for tag, tok in tag_map.items():
        text = text.replace(tag, tok)
    return text


# ---- Unified loss mask (system + user masked, assistant unmasked) ----

def apply_loss_mask(tokens, token_ids, tokenizer, special_map):
    """Return label tensor with system/user/pad masked (-100) and assistant unmasked.
    tokens: list of token strings
    token_ids: list of ints
    special_map: LLAMA_SPECIAL_MAP or GEMMA_SPECIAL_MAP, etc.
    """

    SYS_OPEN = special_map["<sistem>"]
    SYS_CLOSE = special_map["</sistem>"]
    USER_OPEN = special_map["<utilizator>"]
    USER_CLOSE = special_map["</utilizator>"]

    BOS = tokenizer.bos_token or None
    EOS = tokenizer.eos_token or None


    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else -1

    labels = []
    in_system = False
    in_user = False

    for tok, tid in zip(tokens, token_ids):
        # ----- system section -----
        if tok == SYS_OPEN:
            in_system = True
            labels.append(-100)
            continue
        if tok == SYS_CLOSE:
            labels.append(-100)
            in_system = False
            continue

        # ----- user section -----
        if tok == USER_OPEN:
            in_user = True
            labels.append(-100)
            continue
        if tok == USER_CLOSE:
            labels.append(-100)
            in_user = False
            continue

        # Inside system/user → masked
        if in_system or in_user:
            labels.append(-100)
            continue
        # Mask BOS/EOS
        if tok == BOS or tok == EOS:
            labels.append(-100)
            continue

        # Padding should be masked
        if tid == pad_id:
            labels.append(-100)
            continue

        # Otherwise assistant text → unmasked (model predicts these)
        labels.append(tid)

    return labels


# ---- Main preprocessing (LLaMA + Gemma) ----

def make_preprocess(tokenizer):
    family = detect_model_family(tokenizer)
    print(f"Detected model family: {family}")

    special_map = SPECIAL_MAPS.get(family, None)

    def preprocess(example):
        original = example["formatted_text"]

        # Replace <sistem>/<utilizator>/<asistent> tags with model-specific tokens
        if special_map is not None:
            original_local = replace_tags(original, special_map)
        else:
            # If unknown family, just use raw text
            original_local = original

        # Add BOS/EOS (assumes tokenizer has these set correctly)
        text = tokenizer.bos_token + original_local + tokenizer.eos_token

        # Tokenize
        encoded = tokenizer(
            text,
            add_special_tokens=False,
            return_attention_mask=False,
        )
        ids = encoded["input_ids"]

        # Convert ids → tokens for boundary detection
        tokens = tokenizer.convert_ids_to_tokens(ids)

        # Build labels using unified masking logic
        if special_map is not None:
            labels = apply_loss_mask(tokens, ids, tokenizer, special_map)
        else:
            # Fallback: no masking, train on everything except pad
            pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else -1
            labels = [tid if tid != pad_id else -100 for tid in ids]

        # loss_mask: 1 where we compute loss, 0 where ignored
        loss_mask = [0 if l == -100 else 1 for l in labels]


        return {
            "input_ids": ids,
            "labels": labels,
            "loss_mask": loss_mask,
        }

    return preprocess


In [9]:
preprocess_fn = make_preprocess(tokenizer)
tokenized_ds = ds.map(
    preprocess_fn,
    remove_columns=[k for k,v in ds.features.items()]
)

Detected model family: gemma


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [11]:
tokenized_ds

Dataset({
    features: ['input_ids', 'labels', 'loss_mask'],
    num_rows: 100
})

In [12]:
from torch.nn.utils.rnn import pad_sequence
from dataclasses import dataclass

def get_attention_mask_for_packed_sequence(x, token_id, eos: bool = True):
    B, T = x.shape
    eos_idx = (x.view(-1) == token_id).nonzero(as_tuple=True)[0] + eos
    eos_idx_expanded = torch.cat([eos_idx, torch.arange(0,B*T+1,T)]).unique().sort()[0]
    normalized_idx = eos_idx_expanded - (eos_idx_expanded // T) * T
    normalized_idx = torch.where(normalized_idx == 0, T, normalized_idx)
    reps = normalized_idx[1:] - normalized_idx[:-1]
    reps = torch.where(reps < 1, normalized_idx[1:], reps)
    repeated_idx = torch.repeat_interleave(normalized_idx[1:], reps).view(B,1,T).expand(-1,T,-1)
    mask_indices = torch.arange(T).view(1,-1,1).expand(B, -1, T)
    mask = torch.ones(T, T, dtype=torch.bool).tril().expand(B, -1, -1) # SWITCH THIS BACKKK
    mask = mask.masked_fill(mask_indices >= repeated_idx, False)
    #mask = (~mask).float() * -1e9
    mask = mask.unsqueeze(1)


    # get position ids for packed sequence
    pos_ids = (torch.arange(B*T) - torch.repeat_interleave(eos_idx_expanded[:-1], reps)).view(B,T)
    return mask, pos_ids

@dataclass
class PackedSequenceDataCollator:
    tokenizer: any
    pack_length: int
    eos_token_id: int

    def __call__(self, features):
        print("\nInitialising Collator Call..")
        print("Features len:", len(features))

        sequences = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        labels    = [torch.tensor(f["labels"],    dtype=torch.long) for f in features]

        packed_rows_x = []
        packed_rows_y = []

        cur_tokens = []
        cur_labels = []
        cur_len = 0

        for seq, lab in zip(sequences, labels):
            seq_len = len(seq)

            # TRUNCATION RULE
            if seq_len > self.pack_length:
                seq = seq[:self.pack_length]
                lab = lab[:self.pack_length]
                seq[-1] = self.eos_token_id
                lab[-1] = self.eos_token_id
                seq_len = self.pack_length

            if cur_len + seq_len > self.pack_length:
                # flush with exact pack_length padding
                pad_needed = self.pack_length - len(cur_tokens)
                if pad_needed > 0:
                    cur_tokens += [self.eos_token_id] * pad_needed
                    cur_labels += [self.eos_token_id] * pad_needed

                packed_rows_x.append(torch.tensor(cur_tokens))
                packed_rows_y.append(torch.tensor(cur_labels))

                cur_tokens = []
                cur_labels = []
                cur_len = 0

            cur_tokens.extend(seq.tolist())
            cur_labels.extend(lab.tolist())
            cur_len += seq_len

        # flush last row (also exact pack_length)
        if cur_tokens:
            pad_needed = self.pack_length - len(cur_tokens)
            if pad_needed > 0:
                cur_tokens += [self.eos_token_id] * pad_needed
                cur_labels += [self.eos_token_id] * pad_needed

            packed_rows_x.append(torch.tensor(cur_tokens))
            packed_rows_y.append(torch.tensor(cur_labels))

        padded_x = torch.stack(packed_rows_x)
        padded_labels = torch.stack(packed_rows_y)

        mask4, pos_ids = get_attention_mask_for_packed_sequence(
            padded_x, self.eos_token_id
        )

        print("Created mask with shape:", mask4.shape)

        return {
            "input_ids": padded_x,
            "labels": padded_labels,
            "attention_mask": mask4,
            "position_ids": pos_ids,
        }

In [13]:
collator = PackedSequenceDataCollator(
    tokenizer = tokenizer,
    pack_length = 512,
    eos_token_id = tokenizer.eos_token_id
)

In [14]:
import torch
device = "cpu"

if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"

model = model.to(device)

### Test with a Trainer

In [26]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="llama3_dolly_packed",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    num_train_epochs=1,
    fp16=True,
)

trainer = Trainer(
    model = model,
    train_dataset = tokenized_ds,
    data_collator = collator,
    args = training_args
)

In [ ]:
trainer.train()

Getting OOMs with the custom mask approach. Try vanilla way see if still getting OOMs

### Vanilla with Unsloth

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------------------------
# 1. Load model + tokenizer
# ---------------------------
model_name = "meta-llama/Llama-3.2-1B"

model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = model.to('cuda')

from datasets import load_dataset
ds = load_dataset('danp27/norobots_sft')['train'].select(range(100))

# ---------------------------------------------------
# 3. Tag replacement for Llama
# ---------------------------------------------------
reserved = [
    tok.content
    for tok in tokenizer.added_tokens_decoder.values()
    if "reserved_special_token" in tok.content
]

LLAMA_SPECIAL_MAP = {
    "<sistem>": "<|reserved_special_token_0|>",
    "</sistem>": "<|reserved_special_token_1|>",
    "<utilizator>": "<|reserved_special_token_2|>",
    "</utilizator>": "<|reserved_special_token_3|>",
    "<asistent>": "<|reserved_special_token_4|>",
    "</asistent>": "<|reserved_special_token_5|>",
}

def detect_is_llama(tokenizer):
    return "<|reserved_special_token_0|>" in tokenizer.get_vocab()

def replace_tags(text, tag_map):
    for t, tok in tag_map.items():
        text = text.replace(t, tok)
    return text

# ---------------------------------------------------
# 4. Preprocess (NO PACKING, NO CUSTOM MASK)
# ---------------------------------------------------
def make_preprocess(tokenizer):
    IS_LLAMA = detect_is_llama(tokenizer)

    
    
    def preprocess(example):
        text = example["formatted_text"]

        if IS_LLAMA:
            text = replace_tags(text, LLAMA_SPECIAL_MAP)

        text = tokenizer.bos_token + text + tokenizer.eos_token

        enc = tokenizer(
            text,
            add_special_tokens=False,
            return_attention_mask=True,   # <-- keep standard mask
        )

        return {
            "input_ids": enc["input_ids"],
            "labels": enc["input_ids"].copy(),
            "attention_mask": enc["attention_mask"]
        }

    return preprocess

preprocess = make_preprocess(tokenizer)
tokenized_ds = ds.map(preprocess, num_proc=1)


# ---------------------------
# 6. Trainer settings
# ---------------------------
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="llama3_baseline_test",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=1,
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_ds,
    args=training_args,
)

# ---------------------------
# 7. Train (baseline test)
# ---------------------------
trainer.train()


In [8]:
!pip uninstall -y bitsandbytes
!pip install bitsandbytes==0.45.0
!pip install transformers==4.46.3 accelerate -U
!pip install unsloth --force-reinstall


Found existing installation: bitsandbytes 0.48.2
Uninstalling bitsandbytes-0.48.2:
  Successfully uninstalled bitsandbytes-0.48.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 11.4 MB/s eta 0:00:0000:01:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2025.11.3 requires bitsandbytes!=0.46.0,!=0.48.0,>=0.45.5, but you have bitsandbytes 0.45.0 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 76.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the foll

  Using cached unsloth-2025.11.3-py3-none-any.whl.metadata (61 kB)
  Using cached unsloth_zoo-2025.11.4-py3-none-any.whl.metadata (32 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.5 MB/s eta 0:00:00
  Using cached tyro-0.9.35-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
  Using cached trl-0.23.0-py3-none-any.whl.metadata (11 kB)
  Using cached pyarrow-22.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.2 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━

In [9]:
!pip install bitsandbytes --upgrade

In [1]:
# ---------------------------
# 0. Install Unsloth if needed
# ---------------------------
# pip install unsloth[colab]

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

# ---------------------------
# 1. Load Unsloth 4-bit model
# ---------------------------
max_seq_length = 2048
dtype = None          # auto-select
load_in_4bit = True   # qLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",   # similar to your original 1B model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# ---------------------------
# 2. Apply LoRA with Unsloth
# ---------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# ---------------------------
# 3. Load dataset (same as your script)
# ---------------------------
ds = load_dataset("danp27/norobots_sft")["train"].select(range(100))

# ---------------------------
# 4. Tag replacement for Llama
# ---------------------------
LLAMA_SPECIAL_MAP = {
    "<sistem>": "<|reserved_special_token_0|>",
    "</sistem>": "<|reserved_special_token_1|>",
    "<utilizator>": "<|reserved_special_token_2|>",
    "</utilizator>": "<|reserved_special_token_3|>",
    "<asistent>": "<|reserved_special_token_4|>",
    "</asistent>": "<|reserved_special_token_5|>",
}

def detect_is_llama(tokenizer):
    return "<|reserved_special_token_0|>" in tokenizer.get_vocab()

def replace_tags(text, tag_map):
    for t, tok in tag_map.items():
        text = text.replace(t, tok)
    return text

# ---------------------------
# 5. Preprocessing (NO PACKING)
# ---------------------------
def make_preprocess(tokenizer):
    IS_LLAMA = detect_is_llama(tokenizer)

    def preprocess(example):
        text = example["formatted_text"]

        # Replace tags only for LLaMA tokenizer
        if IS_LLAMA:
            text = replace_tags(text, LLAMA_SPECIAL_MAP)

        # Add BOS/EOS
        text = tokenizer.bos_token + text + tokenizer.eos_token

        enc = tokenizer(
            text,
            add_special_tokens=False,
            return_attention_mask=True,
        )

        return {
            "input_ids": enc["input_ids"],
            "labels": enc["input_ids"].copy(),
            "attention_mask": enc["attention_mask"],
        }

    return preprocess

preprocess = make_preprocess(tokenizer)
tokenized_ds = ds.map(preprocess)

# # ---------------------------
# # 6. Prepare for training
# # ---------------------------
# # Unsloth patch for speed (optional but recommended)
# FastLanguageModel.for_training(model)

# # ---------------------------
# # 7. Train with SFTTrainer (Unsloth)
# # ---------------------------
# from trl import SFTTrainer
# from transformers import TrainingArguments

# training_args = TrainingArguments(
#     output_dir="unsloth_llama3_qLoRA",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=2,
#     learning_rate=2e-5,
#     num_train_epochs=1,
#     logging_steps=1,
#     fp16=True,   # HIGHLY recommended
# )

# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=tokenized_ds,
#     dataset_text_field=None,   # we already tokenized manually
#     max_seq_length=max_seq_length,
#     args=training_args,
# )

# trainer.train()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2025.11.3 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# ---------------------------
# 6. Prepare model for training
# ---------------------------
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported
import os, time, numpy as np
from transformers import TrainerCallback
from tqdm import tqdm

FastLanguageModel.for_training(model)

# ---------------------------
# 7. Create fake dataset field required by UnslothTrainer
# ---------------------------
# UnslothTrainer expects dataset_text_field="text".
# We already tokenized, so we add dummy placeholder text.
tokenized_ds = tokenized_ds.add_column("text", [""] * len(tokenized_ds))


# ---------------------------
# 8. Tokens-per-step calculation
# ---------------------------
per_device_train_batch_size = 1
gradient_accumulation_steps = 2
max_seq_length = 2048
tokens_per_step = per_device_train_batch_size * gradient_accumulation_steps * max_seq_length


# ---------------------------
# 9. Timer callback (unchanged)
# ---------------------------
class StepTimerCallback(TrainerCallback):
    def __init__(self, tokens_per_step):
        self.tokens_per_step = tokens_per_step
        self.step_times = []
        self.last_time = None

    def on_step_begin(self, args, state, control, **kwargs):
        self.last_time = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        if self.last_time is None:
            return
        elapsed = time.perf_counter() - self.last_time
        self.step_times.append(elapsed)
        if len(self.step_times) > 5:
            mean_t = np.mean(self.step_times[5:])
            tok_s = self.tokens_per_step / mean_t
            tqdm.write(f"[Step {state.global_step}] {mean_t:.2f}s/step  ({tok_s:,.0f} tok/s)")
        return control


# ---------------------------
# 10. Unsloth TrainingArguments
# ---------------------------
training_args = UnslothTrainingArguments(
    output_dir="unsloth_llama3_qLoRA",
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=1,

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=50,
    seed=3407,

    report_to="wandb",
    run_name=os.environ.get("WANDB_RUN_NAME", "unsloth_run"),
)

# ---------------------------
# 11. UnslothTrainer
# ---------------------------
trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    dataset_text_field="text",     # required, even though unused
    max_seq_length=max_seq_length,
    dataset_num_proc=2,

    args=training_args,
    callbacks=[StepTimerCallback(tokens_per_step)],
)

# ---------------------------
# 12. Train
# ---------------------------
trainer.train()
